In [0]:
%run ../init/kafka_config

In [0]:
from pyspark.sql.functions import (
    col, sum, count, round, avg,
    date_format, hour, dayofweek,
    weekofyear, month, year,
    when, current_timestamp,
    dayofmonth, to_date
)

In [0]:
reporting_schema,catalog_name='reporting','sales_catalog'
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {reporting_schema}")
spark.sql(f"USE SCHEMA {reporting_schema}")

print("✓ Using sales_catalog.reporting")

In [0]:
main_schema='main'
orders_df          = spark.read.table(f"sales_catalog.{main_schema}.orders")
users_df           = spark.read.table(f"sales_catalog.{main_schema}.users")
items_df           = spark.read.table(f"sales_catalog.{main_schema}.items")
inventory_df       = spark.read.table(f"sales_catalog.{main_schema}.inventory")
balance_events_df  = spark.read.table(f"sales_catalog.{main_schema}.balance_events")
inventory_events_df= spark.read.table(f"sales_catalog.{main_schema}.inventory_events")
notifications_df   = spark.read.table(f"sales_catalog.{main_schema}.notifications")
order_logs_df      = spark.read.table(f"sales_catalog.{main_schema}.order_logs")

In [0]:
# calculate total deducted per user
balance_deducted = balance_events_df \
    .groupBy("user_id") \
    .agg(round(sum("amount_change"), 2).alias("total_deducted"))

dim_users = users_df \
    .join(balance_deducted, on="user_id", how="left") \
    .withColumn("total_deducted",
        when(col("total_deducted").isNull(), 0.0)
        .otherwise(col("total_deducted"))
    ) \
    .withColumn("current_balance",
        round(col("balance") + col("total_deducted"), 2)
    ) \
    .withColumn("total_spent",
        round(-col("total_deducted"), 2)
    ) \
    .select(
        col("user_id"),
        col("username"),
        col("email"),
        col("balance").alias("original_balance"),
        col("current_balance"),
        col("total_spent"),
        current_timestamp().alias("last_updated")
    )

dim_users.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.dim_users")


In [0]:
# calculate total deducted per item
stock_deducted = inventory_events_df \
    .groupBy("item_id") \
    .agg(sum("quantity_change").alias("total_deducted"))

dim_items = items_df \
    .join(inventory_df,   on="item_id", how="left") \
    .join(stock_deducted, on="item_id", how="left") \
    .withColumn("total_deducted",
        when(col("total_deducted").isNull(), 0)
        .otherwise(col("total_deducted"))
    ) \
    .withColumn("current_stock",
        col("quantity_available") + col("total_deducted")
    ) \
    .withColumn("total_sold",
        -col("total_deducted")
    ) \
    .select(
        col("item_id"),
        col("item_name"),
        col("price"),
        col("quantity_available").alias("original_stock"),
        col("current_stock"),
        col("total_sold"),
        current_timestamp().alias("last_updated")
    )

dim_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.dim_items")

In [0]:
fact_orders = orders_df \
    .join(users_df.select("user_id","username"),
        on="user_id", how="left") \
    .join(items_df.select("item_id","item_name","price"),
        on="item_id", how="left") \
    .withColumn("total_cost",
        round(col("price") * col("item_quantity"), 2)
    ) \
    .withColumn("date",
        to_date(col("timestamp"))
    ) \
    .withColumn("hour",
        hour(col("timestamp"))
    ) \
    .withColumn("day_of_week",
        date_format(col("timestamp"), "EEEE")
    ) \
    .withColumn("day_number",
        dayofweek(col("timestamp"))
    ) \
    .withColumn("day_of_month",
        dayofmonth(col("timestamp"))
    ) \
    .withColumn("week_number",
        weekofyear(col("timestamp"))
    ) \
    .withColumn("month",
        month(col("timestamp"))
    ) \
    .withColumn("month_name",
        date_format(col("timestamp"), "MMMM")
    ) \
    .withColumn("year",
        year(col("timestamp"))
    ) \
    .select(
        col("order_id"),
        col("user_id"),
        col("username"),
        col("item_id"),
        col("item_name"),
        col("item_quantity"),
        col("price"),
        col("total_cost"),
        col("category"),
        col("status"),
        col("timestamp"),
        col("date"),
        col("hour"),
        col("day_of_week"),
        col("day_number"),
        col("day_of_month"),
        col("week_number"),
        col("month"),
        col("month_name"),
        col("year")
    )

fact_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.fact_orders")


In [0]:
daily_summary = fact_orders \
    .groupBy("date", "year", "month", "month_name", "day_of_month", "day_of_week") \
    .agg(
        count("order_id").alias("total_orders"),
        count(when(col("status") == "ORDER_PLACED", 1)).alias("successful_orders"),
        count(when(col("status") != "ORDER_PLACED", 1)).alias("failed_orders"),
        round(sum(
            when(col("status") == "ORDER_PLACED", col("total_cost"))
            .otherwise(0)
        ), 2).alias("total_revenue"),
        round(avg(
            when(col("status") == "ORDER_PLACED", col("total_cost"))
        ), 2).alias("avg_order_value"),
        col("date").cast("string").alias("date_str")
    ) \
    .orderBy("date")

daily_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.daily_summary")


In [0]:
hourly_summary = fact_orders \
    .groupBy("date", "hour") \
    .agg(
        count("order_id").alias("total_orders"),
        count(when(col("status") == "ORDER_PLACED", 1)).alias("successful_orders"),
        round(sum(
            when(col("status") == "ORDER_PLACED", col("total_cost"))
            .otherwise(0)
        ), 2).alias("total_revenue")
    ) \
    .orderBy("date", "hour")

hourly_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.hourly_summary")

In [0]:
item_summary = fact_orders \
    .filter(col("status") == "ORDER_PLACED") \
    .groupBy("item_id", "item_name", "price") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("item_quantity").alias("total_quantity_sold"),
        round(sum("total_cost"), 2).alias("total_revenue"),
        round(avg("total_cost"), 2).alias("avg_order_value")
    ) \
    .join(
        dim_items.select("item_id","current_stock"),
        on="item_id", how="left"
    ) \
    .orderBy(col("total_revenue").desc())

item_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.item_summary")

In [0]:
user_summary = fact_orders \
    .groupBy("user_id", "username", "category") \
    .agg(
        count("order_id").alias("total_orders"),
        count(when(col("status") == "ORDER_PLACED", 1)).alias("successful_orders"),
        count(when(col("status") != "ORDER_PLACED", 1)).alias("failed_orders"),
        round(sum(
            when(col("status") == "ORDER_PLACED", col("total_cost"))
            .otherwise(0)
        ), 2).alias("total_spent")
    ) \
    .join(
        dim_users.select("user_id","current_balance"),
        on="user_id", how="left"
    ) \
    .orderBy(col("total_spent").desc())

user_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.user_summary")


In [0]:
failure_summary = order_logs_df \
    .filter(col("status") != "ORDER_PLACED") \
    .groupBy("status") \
    .agg(
        count("order_id").alias("total_failures"),
        count("user_id").alias("affected_users")
    ) \
    .orderBy(col("total_failures").desc())

failure_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_catalog.reporting.failure_summary")

In [0]:
print("\n── dim_users ───────────────────────────")
spark.read.table("sales_catalog.reporting.dim_users").show(truncate=False)

print("── dim_items ───────────────────────────")
spark.read.table("sales_catalog.reporting.dim_items").show(truncate=False)

print("── fact_orders ─────────────────────────")
spark.read.table("sales_catalog.reporting.fact_orders").show(truncate=False)

print("── daily_summary ───────────────────────")
spark.read.table("sales_catalog.reporting.daily_summary").show(truncate=False)

print("── hourly_summary ──────────────────────")
spark.read.table("sales_catalog.reporting.hourly_summary").show(truncate=False)

print("── item_summary ────────────────────────")
spark.read.table("sales_catalog.reporting.item_summary").show(truncate=False)

print("── user_summary ────────────────────────")
spark.read.table("sales_catalog.reporting.user_summary").show(truncate=False)

print("── failure_summary ─────────────────────")
spark.read.table("sales_catalog.reporting.failure_summary").show(truncate=False)

print("\n✓ All reporting tables created successfully")
print("Schema: sales_catalog.reporting")
